# BeltCrackDet on BeltCrack9kd: replication in Colab

Runs the **authors' own code** ([UESTC-nnLab/BeltCrack](https://github.com/UESTC-nnLab/BeltCrack),
pinned commit) with their settings: 5-frame BeltCrackDet, 512×512, SGD lr 0.01 cosine, 100 epochs,
batch 4, YOLOX-s pretrained weights. As in their 9kd config, the test split is used for validation
during training (9kd has no val split).

**What you do**
1. Download **BeltCrack9kd** from https://doi.org/10.57760/sciencedb.31181 (unzip password `cv205`).
2. Put it in the Google Drive folder set in section 1 (`MyDrive/BeltCrack` by default), extracted or still packed.
   Faster option: run `prepare_beltcrack9kd.py` on your own PC and upload only the
   `BeltCrack9kd_prep.zip` it makes (about 1 GB).
3. `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

Training takes many hours and spans several Colab sessions. When a session ends, reconnect and
`Run all` again: it resumes from the last finished epoch saved on Drive.

**Reference:** the paper's full model reaches **30.79 mAP50** on the 9kd test split.

**What differs from the authors' setup, and why**
- Frames are pre-resized once, so that 2 CPU cores can feed the GPU. Train frames are stored at the
  exact size the authors' loader resizes to, with the same int() box rounding, so training sees the same
  pixels apart from JPEG re-encoding at quality 95. Test frames are kept at 1024 px on the long side.
- One GPU instead of two (same total batch of 4). Kaggle with 2×T4 uses both GPUs like the authors.
- The checkpoint is evaluated as *last epoch* by default. The authors pick a checkpoint by looking at
  results, which on 9kd means looking at the test set; that option is in section 10, clearly labelled.

## 1. Settings

In [ ]:
DRIVE_FOLDER    = "BeltCrack"      # folder in My Drive with the dataset (images + annotation files, or archives)
UNZIP_PASSWORD  = "cv205"
ARCHIVE_PATTERN = "*.rar"          # archives to extract, e.g. BeltCrack14ks.part01.rar ... part08.rar.
                                   # The data is identified by its contents, not the file name. Other zips
                                   # in the folder (beltcrack_yolo960.zip, ...) are left alone.

EPOCHS          = 100     # paper: 100
BATCH           = 4       # paper: 4. Lowered automatically if it does not fit in GPU memory
FP16            = False   # paper: False. True is faster on a T4 but deviates from the paper

EVAL_CONF       = 0.003   # values in the authors' vid_map_coco.py (the paper text says 0.001 / 0.65)
EVAL_NMS        = 0.20
EVAL_CHECKPOINT = "last"  # "last", or a checkpoint file name such as "ep0050-loss....pth"

REPO_URL        = "https://github.com/UESTC-nnLab/BeltCrack.git"
REPO_COMMIT     = "0d1b24d"   # the version this notebook's patches were written against

## 2. Environment, Drive and the authors' code

In [ ]:
%pip install -q pycocotools PyWavelets einops

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/working").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE, STORE = Path("/content"), Path("/content/drive/MyDrive") / DRIVE_FOLDER
    SEARCH = [STORE]
elif IN_KAGGLE:
    BASE = Path("/kaggle/working")
    STORE = BASE / DRIVE_FOLDER
    SEARCH = [STORE, Path("/kaggle/input")]
else:
    BASE = Path.cwd()
    STORE = BASE / DRIVE_FOLDER
    SEARCH = [STORE]

STORE.mkdir(parents=True, exist_ok=True)
RUNS = STORE / "runs"                      # checkpoints and logs, kept on Drive
RUNS.mkdir(exist_ok=True)
REPO = BASE / "BeltCrack_code"           # the authors' repository
PREP = BASE / "BeltCrack9kd_prep"
PREP_ZIP = "BeltCrack9kd_prep.zip"
WORKERS = min(8, os.cpu_count() or 2)
assert " " not in str(PREP), "the authors' annotation format cannot hold paths with spaces"


def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, **kw)
    if r.returncode:
        raise RuntimeError(f"{cmd} failed:\n{r.stdout[-1500:]}\n{r.stderr[-1500:]}")
    return r.stdout


if not (REPO / ".git").exists():
    sh(["git", "clone", "-q", REPO_URL, str(REPO)])
sh(["git", "-C", str(REPO), "checkout", "-q", "-f", REPO_COMMIT])   # clean copy before patching
sh(["git", "-C", str(REPO), "clean", "-q", "-fd", "-e", "map_out", "-e", ".temp*"])

try:
    import torch
    print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name}, {p.total_memory / 2**30:.1f} GB")
    if not torch.cuda.is_available():
        print("WARNING: no GPU. On Colab: Runtime > Change runtime type > T4 GPU.")
except ImportError:
    print("torch is not installed")
print("storage:", STORE)

## 3. Dataset preparation code
The same code is in `prepare_beltcrack9kd.py`, for running on your own PC.

In [ ]:
# ---------------------------------------------------------------------------------------
# BeltCrack9kd preparation for the authors' BeltCrackDet code.
#
# Train frames are stored at exactly the size the authors' loader resizes them to
# (letterbox scale to 512, int() sizes, bicubic), so their loader's own resize becomes a
# no-op and training sees the same pixels as on full-resolution files (apart from JPEG
# re-encoding at quality 95). Box coordinates use the same int() truncation as the loader.
# Test frames are kept at 1024 on the long side, because the authors' test script
# stretches each frame to 512x512 without letterbox.
# ---------------------------------------------------------------------------------------
import json
import os
import re
import shutil
import subprocess
import zipfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from PIL import Image

TRAIN_SIZE = 512
TEST_LONG_SIDE = 1024
JPEG_QUALITY = 95
HISTORY = 5                      # frames per sample in the authors' model (key frame + 4 before)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
BICUBIC = getattr(Image, "Resampling", Image).BICUBIC
NUM = r"-?\d+(?:\.\d+)?"
LINE_RE = re.compile(rf"^(\S+?\.(?:jpg|jpeg|png|bmp))((?:\s+{NUM},{NUM},{NUM},{NUM},{NUM})*)\s*$", re.I)


# ----------------------------------- archives -----------------------------------------
def is_first_volume(p):
    """The archive file to hand to the extractor; later parts of multi-part sets are skipped."""
    n = p.name.lower()
    m = re.search(r"\.part(\d+)\.rar$", n)
    if m:
        return int(m.group(1)) == 1
    m = re.search(r"\.(?:zip|7z)\.(\d+)$", n)
    if m:
        return int(m.group(1)) == 1
    return bool(re.search(r"\.(zip|7z|rar|tar|tgz|tar\.gz)$", n))


def find_archives(folders, skip_names=(), pattern="*"):
    """First volumes of archives under folders whose file name matches pattern, e.g. "*.rar"."""
    found = []
    for folder in folders:
        folder = Path(folder)
        if not folder.exists():
            continue
        for p in sorted(folder.rglob(pattern)):
            if p.is_file() and is_first_volume(p) and p.name not in skip_names and "runs" not in p.parts:
                found.append(p)
    return found


def extract_archive(archive, out, password):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    print(f"extracting {archive.name} ...", flush=True)
    low = archive.name.lower()
    unrar = shutil.which("unrar")
    seven = shutil.which("7z") or shutil.which("7zz") or shutil.which("7za")
    if low.endswith(".rar") and unrar:
        cmd, ok = [unrar, "x", "-o+", "-inul", f"-p{password}", str(archive), str(out) + os.sep], (0,)
    elif seven:
        cmd, ok = [seven, "x", "-y", "-bso0", "-bsp0", f"-p{password}", f"-o{out}", str(archive)], (0, 1)
    elif low.endswith(".zip"):
        with zipfile.ZipFile(archive) as z:
            z.extractall(out, pwd=password.encode())
        return
    elif low.endswith((".tar", ".tgz", ".tar.gz")):
        shutil.unpack_archive(str(archive), str(out))
        return
    else:
        raise RuntimeError(f"No extractor for {archive.name}: install unrar or 7-Zip "
                           "(Colab/Ubuntu: apt-get install -y unrar p7zip-full).")
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode not in ok:
        raise RuntimeError(f"Extracting {archive.name} failed (exit {r.returncode}). Check the password and "
                           f"that every part of a multi-part archive is present.\n{r.stdout[-800:]}{r.stderr[-800:]}")


def extract_all(folders, out, password, skip_names=(), pattern="*"):
    archives = find_archives(folders, skip_names, pattern)
    for a in archives:
        extract_archive(a, out, password)
    for a in find_archives([out]):                 # archives packed inside the archive
        extract_archive(a, a.parent, password)
        a.unlink()
    return archives


# Labelled images per split in the authors' annotation files.
KNOWN_SPLITS = {"BeltCrack9kd": {"train": 7697, "test": 1948},
                "BeltCrack14ks": {"train": 8773, "test": 2718}}


def check_dataset(info, expect="BeltCrack9kd"):
    """Identify the data by its split sizes (archive names are not reliable). Stops on the other dataset."""
    # images without boxes are left out of the lists (as coco_to_txt.py does) but count towards the published sizes
    got = {sp: info[sp]["images"] + info[sp].get("without_boxes", 0) for sp in ("train", "test")}
    for name, ref in KNOWN_SPLITS.items():
        if got == ref:
            if name != expect:
                raise RuntimeError(f"These annotations are {name} ({got}), not {expect}.")
            return f"OK: matches {name} ({got['train']} train / {got['test']} test images)"
    return (f"WARNING: {got['train']} train / {got['test']} test images do not match the published "
            f"{expect} counts {KNOWN_SPLITS[expect]}. Check the annotation files listed above.")


def locate_root(base):
    return Path(base)          # the whole folder is searched for images and annotation files


# ----------------------------------- annotations --------------------------------------
class PathIndex:
    """Matches paths written on the authors' machine (/home/hjh/BeltCrack9kd/...) to local files."""

    def __init__(self, files, max_k=6):
        self.max_k = max_k
        self.idx = [defaultdict(list) for _ in range(max_k + 1)]
        for p in files:
            parts = p.parts
            for k in range(1, min(max_k, len(parts)) + 1):
                self.idx[k]["/".join(parts[-k:])].append(p)

    def resolve(self, name):
        parts = [x for x in str(name).replace("\\", "/").split("/") if x]
        for k in range(min(self.max_k, len(parts)), 0, -1):
            hits = self.idx[k].get("/".join(parts[-k:]))
            if hits:
                return hits[0] if len(hits) == 1 else None
        return None

    def exact(self, parts):
        """The one file whose last len(parts) path parts are exactly parts, else None (no shorter fallback)."""
        if not 1 <= len(parts) <= self.max_k:
            return None
        hits = self.idx[len(parts)].get("/".join(parts))
        return hits[0] if hits and len(hits) == 1 else None


def split_of(rel):
    for part in reversed(Path(rel).parts):
        low = part.lower()
        for key in ("train", "test", "val"):
            if key in low:
                return key
    return None


def frame_no(p):
    m = re.search(r"(\d+)$", Path(p).stem)
    return int(m.group(1)) if m else None


def read_txt(path, index):
    entries, unresolved = [], 0
    for line in Path(path).read_text(encoding="utf-8", errors="ignore").splitlines():
        m = LINE_RE.match(line.strip())
        if not m:
            continue
        p = index.resolve(m.group(1))
        if p is None:
            unresolved += 1
            continue
        boxes = [tuple(int(float(v)) for v in b.split(",")) for b in m.group(2).split()]
        entries.append((p, boxes))
    return entries, unresolved


def load_json(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return None


def is_coco(data):
    return isinstance(data, dict) and "images" in data and "annotations" in data


def is_labelme(data):
    return isinstance(data, dict) and isinstance(data.get("shapes"), list)


def labelme_image(path, data, index, image_set):
    """The image a LabelMe file belongs to: same folder and stem, then imagePath, then <split>/<seq>/<n>.<ext>."""
    path = Path(path)
    exts = sorted(IMG_EXT)
    for ext in exts:
        for p in (path.with_suffix(ext), path.with_suffix(ext.upper())):
            if p in image_set:
                return p
    ip = data.get("imagePath")
    if ip:
        p = Path(os.path.normpath(path.parent / str(ip).replace("\\", "/")))
        if p in image_set:
            return p
    if len(path.parts) >= 3:
        split, seq = path.parts[-3], path.parts[-2]
        for ext in exts:
            for e in (ext, ext.upper()):
                p = index.exact([split, seq, path.stem + e])
                if p is not None:
                    return p
    return None


def labelme_record(path, data, index, image_set, stats):
    """One LabelMe file as a COCO-style record (what the authors' LabelMe -> COCO step gives), or None if
    no image matches. Box per shape with >= 2 points: x, y = min; w, h = max - min, as floats."""
    p = labelme_image(path, data, index, image_set)
    if p is None:
        return None
    anns = []
    for s in data["shapes"]:
        stats["labels"][str(s.get("label"))] += 1
        stats["shape_types"][str(s.get("shape_type"))] += 1
        pts = s.get("points") or []
        if len(pts) < 2:
            stats["short_shapes"] += 1
            continue
        xs, ys = [float(q[0]) for q in pts], [float(q[1]) for q in pts]
        x, y = min(xs), min(ys)
        anns.append({"bbox": [x, y, max(xs) - x, max(ys) - y], "category_id": 1, "iscrowd": 0})
    return {"path": p, "anns": anns}


def read_coco(path, index, data=None):
    if data is None:
        data = load_json(path)
    if not is_coco(data):
        return None
    recs, unresolved = {}, 0
    for im in data["images"]:
        p = index.resolve(im.get("file_name", ""))
        if p is None:
            unresolved += 1
            continue
        recs[im["id"]] = {"id": im["id"], "path": p, "anns": []}
    for a in data["annotations"]:
        r = recs.get(a.get("image_id"))
        if r is not None:
            r["anns"].append({"bbox": [float(v) for v in a["bbox"]],
                              "category_id": int(a.get("category_id", 1)),
                              "iscrowd": int(a.get("iscrowd", 0))})
    cats = data.get("categories") or [{"id": 1, "name": "crack"}]
    return list(recs.values()), unresolved, cats


def coco_to_lines(recs):
    """Same rule as the authors' utils_coco/coco_to_txt.py: annotated images only, int() corners."""
    lines = []
    for r in recs:
        if not r["anns"]:
            continue
        boxes = []
        for a in r["anns"]:
            x, y, w, h = a["bbox"]
            x0, y0 = int(x), int(y)
            boxes.append((x0, y0, x0 + int(w), y0 + int(h), a["category_id"] - 1))
        lines.append((r["path"], boxes))
    return lines


def lines_to_coco(lines):
    recs = []
    for i, (p, boxes) in enumerate(lines, 1):
        recs.append({"id": i, "path": p, "anns": [
            {"bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "category_id": c + 1, "iscrowd": 0}
            for x1, y1, x2, y2, c in boxes]})
    return recs


# ----------------------------------- resizing -----------------------------------------
def target_size(iw, ih, mode):
    if mode == "train":                              # same arithmetic as utils/dataloader_for_CRACK.py
        scale = min(TRAIN_SIZE / iw, TRAIN_SIZE / ih)
        return int(iw * scale), int(ih * scale)
    s = min(1.0, TEST_LONG_SIDE / max(iw, ih))
    return max(1, int(round(iw * s))), max(1, int(round(ih * s)))


def resize_job(job):
    src, dst, mode = job
    with Image.open(src) as im:
        im.load()
        iw, ih = im.size
        nw, nh = target_size(iw, ih, mode)
        if im.mode != "RGB":
            im = im.convert("RGB")
        if (nw, nh) != (iw, ih):
            im = im.resize((nw, nh), BICUBIC)
        Path(dst).parent.mkdir(parents=True, exist_ok=True)
        im.save(dst, quality=JPEG_QUALITY, subsampling=0)
    return src, (iw, ih, nw, nh)


def _seq_id(rel, used):
    base = re.sub(r"[^A-Za-z0-9_-]+", "_", rel.as_posix()).strip("_") or "seq"
    sid, k = base, 2
    while sid in used:
        sid, k = f"{base}_{k}", k + 1
    used.add(sid)
    return sid


# ----------------------------------- main ----------------------------------------------
def prepare(root, out, workers=None, log=print):
    root, out = Path(root), Path(out)
    skip = {"runs", out.name}  # earlier training outputs and this script's own output
    images = [p for p in root.rglob("*")
              if p.is_file() and p.suffix.lower() in IMG_EXT and not skip & set(p.relative_to(root).parts)]
    if not images:
        raise RuntimeError(f"No images found under {root}")
    index = PathIndex(images)
    log(f"{len(images)} image files under {root}")

    txt, coco, unresolved, found = defaultdict(list), {}, defaultdict(int), []
    cats = [{"id": 1, "name": "crack"}]
    for f in sorted(root.rglob("*.txt")):
        entries, unres = read_txt(f, index)
        if not entries and not unres:
            continue
        sp = split_of(f.relative_to(root))
        found.append(f"{f.relative_to(root)}  [{sp}] {len(entries)} lines, {unres} unmatched")
        if sp:
            txt[sp] += entries
            unresolved[sp] += unres
    image_set = set(images)
    lm = defaultdict(list)                                   # split -> LabelMe records (one JSON per image)
    lm_stats = {"labels": defaultdict(int), "shape_types": defaultdict(int), "short_shapes": 0}
    lm_count = defaultdict(lambda: defaultdict(int))
    for f in sorted(root.rglob("*.json")):
        data = load_json(f)                                  # each file is parsed once
        sp = split_of(f.relative_to(root))
        if is_labelme(data):
            r = labelme_record(f, data, index, image_set, lm_stats)
            c = lm_count[sp]
            if r is None:
                c["unmatched"] += 1
            elif r["anns"]:
                c["with_boxes"] += 1
                c["boxes"] += len(r["anns"])
                lm[sp].append(r)
            else:
                c["without_boxes"] += 1
                lm[sp].append(r)
            continue
        res = read_coco(f, index, data)
        if res is None:
            continue
        recs, unres, c = res
        found.append(f"{f.relative_to(root)}  [{sp}] {len(recs)} images, {unres} unmatched")
        if sp and sp not in coco:
            coco[sp], cats = recs, c
            unresolved[sp] += unres
    for sp in ("train", "val", "test", None):
        if sp in lm_count:
            c = lm_count[sp]
            found.append(f"LabelMe json [{sp}] {c['with_boxes']} images with boxes, {c['boxes']} boxes, "
                         f"{c['without_boxes']} images without boxes, {c['unmatched']} unmatched"
                         + ("  (merged into train)" if sp == "val" else ""))
    for line in found:
        log("  annotation file: " + line)
    if lm_count:
        log(f"  LabelMe shape labels: {dict(lm_stats['labels'])}  shape_types: {dict(lm_stats['shape_types'])}")

    # The paper's 9kd train set is 33 sequences = the archive's 23 train + 10 val folders; test stays test.
    lm_train = lm["train"] + lm["val"]
    for i, r in enumerate(lm["test"], 1):
        r["id"] = i
    without = {"train": 0, "test": 0}
    if txt["train"]:
        train = txt["train"]
    elif coco.get("train"):
        train = coco_to_lines(coco["train"])
        without["train"] = sum(not r["anns"] for r in coco["train"])
    else:
        train = coco_to_lines(lm_train)
        without["train"] = sum(not r["anns"] for r in lm_train)
        unresolved["train"] += lm_count["train"]["unmatched"] + lm_count["val"]["unmatched"]
    if txt["test"]:
        test_lines = txt["test"]
        test_coco = coco.get("test") or lines_to_coco(test_lines)
    elif coco.get("test"):
        test_lines, test_coco = coco_to_lines(coco["test"]), coco["test"]
        without["test"] = sum(not r["anns"] for r in coco["test"])
    else:
        test_lines = coco_to_lines(lm["test"])
        test_coco = lm["test"] or lines_to_coco(test_lines)  # COCO test file keeps images without boxes
        without["test"] = sum(not r["anns"] for r in lm["test"])
        unresolved["test"] += lm_count["test"]["unmatched"]
    if not train or not test_lines:
        raise RuntimeError("Could not find both train and test annotations. Files found:\n  " + "\n  ".join(found))
    notes = []
    if lm_count["val"] and not (txt["train"] or coco.get("train")):
        notes.append(f"LabelMe val ({len(lm['val'])} images) merged into train: the paper's 9kd train set is "
                     "the 23 train + 10 val sequences.")
    if lm_stats["short_shapes"]:
        notes.append(f"{lm_stats['short_shapes']} LabelMe shapes with fewer than 2 points were skipped.")
    if txt["val"] or coco.get("val"):
        notes.append("A val annotation file exists but is not used: the authors' 9kd config validates on test.")
    for sp, n in unresolved.items():
        if n:
            notes.append(f"{n} {sp} annotation entries did not match an image file and were skipped.")
    n_listed = len(train) + len(test_lines)
    if sum(unresolved.values()) > 0.01 * max(1, n_listed):
        raise RuntimeError("More than 1% of annotation entries match no image file:\n  " + "\n  ".join(notes))

    # folders, frame numbers and the history frames each sample needs
    mode = {}
    for p, _ in train:
        mode.setdefault(p.parent, "train")
    shared = set()
    for p in [p for p, _ in test_lines] + [r["path"] for r in test_coco]:
        if mode.get(p.parent) == "train":
            shared.add(p.parent)
        mode[p.parent] = "test"
    if shared:
        notes.append(f"{len(shared)} sequence folders appear in both train and test.")

    frames = defaultdict(dict)
    for p in images:
        if p.parent in mode:
            n = frame_no(p)
            if n is not None and (n not in frames[p.parent] or p.suffix.lower() == ".jpg"):
                frames[p.parent][n] = p
    listed = [p for p, _ in train] + [p for p, _ in test_lines] + [r["path"] for r in test_coco]
    bad = [p for p in listed if frame_no(p) is None]
    if bad:
        raise RuntimeError(f"{len(bad)} labelled images have no frame number in their name, e.g. {bad[0]}")

    used, seq = set(), {}
    for folder in sorted(mode):
        seq[folder] = _seq_id(folder.relative_to(root), used)

    jobs, gaps = {}, 0
    for p in listed:
        folder, n = p.parent, frame_no(p)
        for k in range(HISTORY):
            m = max(n - k, 0)                       # same clamp as the authors' loaders
            dst = out / "frames" / seq[folder] / f"{m}.jpg"
            if dst in jobs and not (k == 0 and jobs[dst][0] != p):
                continue
            src = p if k == 0 else frames[folder].get(m)
            if src is None:                       # the authors' loader would crash here
                gaps += 1
                src = frames[folder][min(frames[folder], key=lambda x: (abs(x - m), x > m))]
            jobs[dst] = (src, dst, mode[folder])
    if gaps:
        notes.append(f"{gaps} history frames were missing on disk and were filled with the nearest frame.")

    log(f"resizing {len(jobs)} frames ...")
    sizes = {}
    with ThreadPoolExecutor(max_workers=workers or os.cpu_count()) as ex:
        for i, (src, size) in enumerate(ex.map(resize_job, jobs.values(), chunksize=16), 1):
            sizes[src] = size
            if i % 1000 == 0:
                log(f"  {i}/{len(jobs)}")

    def rel(p):
        return f"frames/{seq[p.parent]}/{frame_no(p)}.jpg"

    ann = out / "annotations"
    ann.mkdir(parents=True, exist_ok=True)
    n_train_boxes = 0
    with open(ann / "train.txt", "w") as f:
        for p, boxes in train:
            iw, ih, nw, nh = sizes[p]
            out_boxes = [(int(x1 * nw / iw), int(y1 * nh / ih), int(x2 * nw / iw), int(y2 * nh / ih), c)
                         for x1, y1, x2, y2, c in boxes]
            n_train_boxes += len(out_boxes)
            f.write(rel(p) + "".join(" %d,%d,%d,%d,%d" % b for b in out_boxes) + "\n")
    n_test_boxes = 0
    with open(ann / "test.txt", "w") as f:
        for p, boxes in test_lines:
            iw, ih, nw, nh = sizes[p]
            sx, sy = nw / iw, nh / ih
            out_boxes = [(int(round(x1 * sx)), int(round(y1 * sy)), int(round(x2 * sx)), int(round(y2 * sy)), c)
                         for x1, y1, x2, y2, c in boxes]
            n_test_boxes += len(out_boxes)
            f.write(rel(p) + "".join(" %d,%d,%d,%d,%d" % b for b in out_boxes) + "\n")
    cj = {"info": {"description": "BeltCrack9kd test split, rescaled for BeltCrackDet replication"},
          "licenses": [], "categories": cats, "images": [], "annotations": []}
    for r in test_coco:
        iw, ih, nw, nh = sizes[r["path"]]
        sx, sy = nw / iw, nh / ih
        cj["images"].append({"id": r["id"], "file_name": rel(r["path"]), "width": nw, "height": nh})
        for a in r["anns"]:
            x, y, w, h = a["bbox"]
            cj["annotations"].append({"id": len(cj["annotations"]) + 1, "image_id": r["id"],
                                      "category_id": a["category_id"], "iscrowd": a["iscrowd"],
                                      "bbox": [x * sx, y * sy, w * sx, h * sy], "area": w * sx * h * sy})
    (ann / "test_coco.json").write_text(json.dumps(cj))

    src_boxes = {"train": sum(len(b) for _, b in train), "test": sum(len(b) for _, b in test_lines)}
    assert n_train_boxes == src_boxes["train"] and n_test_boxes == src_boxes["test"]
    info = {
        "source_root": str(root),
        "train": {"images": len(train), "boxes": n_train_boxes, "sequences": len({p.parent for p, _ in train}),
                  "without_boxes": without["train"]},
        "test": {"images": len(test_lines), "boxes": n_test_boxes, "without_boxes": without["test"],
                 "sequences": len({p.parent for p, _ in test_lines}),
                 "coco_images": len(cj["images"]), "coco_boxes": len(cj["annotations"])},
        "frames_written": len(jobs),
        "source_sizes": sorted({f"{s[0]}x{s[1]}" for s in sizes.values()}),
        "train_frame_rule": f"letterbox-exact {TRAIN_SIZE}", "test_frame_rule": f"long side {TEST_LONG_SIDE}",
        "jpeg_quality": JPEG_QUALITY, "notes": notes,
    }
    (out / "prep_info.json").write_text(json.dumps(info, indent=1))
    for n in notes:
        log("NOTE: " + n)
    return info


def make_zip(folder, zip_path):
    folder, zip_path = Path(folder), Path(zip_path)
    tmp = zip_path.with_suffix(".zip.tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as z:
        for p in sorted(folder.rglob("*")):
            if p.is_file():
                z.write(p, Path(folder.name) / p.relative_to(folder))
    os.replace(tmp, zip_path)
    return zip_path


## 4. Get the prepared dataset
Uses `BeltCrack9kd_prep.zip` from Drive if it is there; otherwise extracts the raw archive and prepares it (then saves the zip to Drive, so later sessions skip this).

In [ ]:
if not (PREP / "prep_info.json").exists():
    zips = [d / PREP_ZIP for d in SEARCH if (d / PREP_ZIP).exists()]
    zips += [p for d in SEARCH if d.exists() for p in d.rglob(PREP_ZIP)]
    if zips:
        local = BASE / PREP_ZIP
        if zips[0] != local:
            print("copying", zips[0], "...")
            shutil.copy(zips[0], local)
        with zipfile.ZipFile(local) as z:
            z.extractall(BASE)
        local.unlink()
    else:
        if IN_COLAB and not (shutil.which("unrar") and shutil.which("7z")):
            sh("apt-get -qq update && apt-get -qq install -y unrar p7zip-full")
        raw = BASE / "raw9kd"
        archives = extract_all(SEARCH, raw, UNZIP_PASSWORD, skip_names={PREP_ZIP}, pattern=ARCHIVE_PATTERN)
        print("extracted:", [a.name for a in archives] or "nothing; using an extracted folder")
        if archives:
            root = locate_root(raw)
        else:                                     # an already-extracted dataset folder
            root = locate_root(next(d for d in SEARCH if d.exists()))
        print("dataset root:", root)
        prepare(root, PREP, workers=WORKERS)
        print("saving", PREP_ZIP, "to", STORE, "...")
        z = make_zip(PREP, BASE / PREP_ZIP)
        shutil.copy(z, STORE / PREP_ZIP)
        z.unlink()
        shutil.rmtree(raw, ignore_errors=True)

INFO = json.loads((PREP / "prep_info.json").read_text())
print(check_dataset(INFO))
print(json.dumps({k: INFO[k] for k in ("train", "test", "frames_written", "source_sizes", "notes")}, indent=1))

## 5. Point the authors' scripts at this data
Small, checked patches: each one must match exactly once, or the cell stops. Nothing in the model, loss, optimiser or schedule is changed.

In [ ]:
def _read(p):
    return Path(p).read_text(encoding="utf-8").replace("\r\n", "\n")


def _sub_one(src, pattern, repl, what):
    new, n = re.subn(pattern, repl, src, flags=re.M)
    if n != 1:
        raise RuntimeError(f"patch '{what}': expected 1 match, found {n}. Use REPO_COMMIT = '0d1b24d'.")
    return new


def set_assign(src, name, expr):
    # a plain assignment line only (keyword arguments in calls contain commas and are left alone)
    return _sub_one(src, rf"^([ \t]*){re.escape(name)}[ \t]*=(?!=)[^,\n]*$", lambda m: f"{m.group(1)}{name} = {expr}", name)


def patch_repo(repo):
    repo = Path(repo)
    # train_BeltCrackDet.py: paths, epochs, batch, workers and resume come from environment variables
    f = repo / "train_BeltCrackDet.py"
    s = _read(f)
    for name, expr in [
        ("classes_path", "'model_data/crack.txt'"),
        ("model_path", "os.environ['BC_MODEL_PATH']"),
        ("Init_Epoch", "int(os.environ['BC_INIT_EPOCH'])"),
        ("Freeze_Epoch", "int(os.environ['BC_EPOCHS'])"),
        ("UnFreeze_Epoch", "int(os.environ['BC_EPOCHS'])"),
        ("Freeze_batch_size", "int(os.environ['BC_BATCH'])"),
        ("Unfreeze_batch_size", "int(os.environ['BC_BATCH'])"),
        ("fp16", "os.environ['BC_FP16'] == '1'"),
        ("num_workers", "int(os.environ['BC_WORKERS'])"),
        ("save_dir", "os.environ['BC_SAVE_DIR']"),
        ("train_annotation_path", "os.environ['BC_TRAIN_TXT']"),
        ("val_annotation_path", "os.environ['BC_VAL_TXT']"),
    ]:
        s = set_assign(s, name, expr)
    f.write_text(s)

    # utils_fit.py: save the last checkpoint atomically and record its epoch, for resuming
    f = repo / "utils" / "utils_fit.py"
    s = _read(f)
    s = _sub_one(s, r'^([ \t]*)torch\.save\(save_state_dict, os\.path\.join\(save_dir, "last_epoch_weights\.pth"\)\)[ \t]*$',
                 lambda m: (f'{m.group(1)}_tmp = os.path.join(save_dir, "last_epoch_weights.pth.tmp")\n'
                            f'{m.group(1)}torch.save(save_state_dict, _tmp)\n'
                            f'{m.group(1)}os.replace(_tmp, os.path.join(save_dir, "last_epoch_weights.pth"))\n'
                            f'{m.group(1)}with open(os.path.join(save_dir, "last_epoch.txt"), "w") as _f:\n'
                            f'{m.group(1)}    _f.write(str(epoch + 1))'), "last checkpoint")
    f.write_text(s)

    # callbacks.py: unique image ids for any folder depth, and the epoch number in epoch_map.txt
    f = repo / "utils" / "callbacks.py"
    s = _read(f)
    old_id = '"-".join(line[0].split("/")[6:8]).split(\'.\')[0]'
    s = _sub_one(s, rf"^([ \t]*)image_id[ \t]*=[ \t]*{re.escape(old_id)}[ \t]*$",
                 lambda m: m.group(1) + 'image_id    = line[0].strip("/").replace("/", "-").rsplit(".", 1)[0]', "image id")
    s = _sub_one(s, r'^([ \t]*)f\.write\(str\(temp_map\)\+" "\+str\(pr\)\+" "\+str\(re\)\+" "\+str\(F1\)\)[ \t]*$',
                 lambda m: f'{m.group(1)}f.write(str(epoch)+" "+str(temp_map)+" "+str(pr)+" "+str(re)+" "+str(F1))',
                 "epoch map line")
    f.write_text(s)

    # vid_map_coco.py: test annotations, images, weights and thresholds
    f = repo / "vid_map_coco.py"
    s = _read(f)
    s = set_assign(s, "cocoGt_path", "os.environ['BC_TEST_JSON']")
    s = set_assign(s, "dataset_img_path", "os.environ['BC_DATA_ROOT']")
    s = set_assign(s, "temp_save_path", "os.environ['BC_EVAL_OUT']")
    s = _sub_one(s, r'^([ \t]*)"model_path"[ \t]*:.*$',
                 lambda m: f'{m.group(1)}"model_path": os.environ["BC_EVAL_WEIGHTS"],',
                 "eval weights")
    s = _sub_one(s, r"^([ \t]*)yolo[ \t]*=[ \t]*MAP_vid\(.*\)[ \t]*$",
                 lambda m: (f"{m.group(1)}yolo = MAP_vid(confidence = float(os.environ['BC_EVAL_CONF']), "
                            f"nms_iou = float(os.environ['BC_EVAL_NMS']))"), "eval thresholds")
    f.write_text(s)

    for f in ("train_BeltCrackDet.py", "utils/utils_fit.py", "utils/callbacks.py", "vid_map_coco.py"):
        compile(_read(repo / f), f, "exec")


def write_abs_list(rel_txt, abs_txt, root):
    out = []
    for line in Path(rel_txt).read_text().splitlines():
        if line.strip():
            rel, *boxes = line.split()
            out.append(" ".join([(Path(root) / rel).as_posix(), *boxes]))
    Path(abs_txt).write_text("\n".join(out) + "\n")
    return len(out)


def find_resume(runs):
    # Most advanced finished epoch across all sessions, and its weights.
    best = (0, None)
    for f in Path(runs).glob("loss_*/last_epoch.txt"):
        w = f.parent / "last_epoch_weights.pth"
        txt = f.read_text().strip()
        if w.exists() and txt.isdigit() and int(txt) > best[0]:
            best = (int(txt), w)
    return best


def read_epoch_maps(runs):
    # epoch -> (mAP50, precision, recall, F1) from the training callback, latest session wins.
    rows = {}
    for f in sorted(Path(runs).glob("loss_*/epoch_map.txt"), key=lambda p: p.parent.name):
        for line in f.read_text().splitlines():
            parts = line.split()
            if len(parts) == 5:
                rows[int(parts[0])] = tuple(float(x) for x in parts[1:])
    return dict(sorted(rows.items()))


patch_repo(REPO)
TRAIN_TXT = PREP / "annotations" / "train_abs.txt"
TEST_TXT = PREP / "annotations" / "test_abs.txt"
n_tr = write_abs_list(PREP / "annotations" / "train.txt", TRAIN_TXT, PREP)
n_te = write_abs_list(PREP / "annotations" / "test.txt", TEST_TXT, PREP)
os.environ.update({
    "BC_TRAIN_TXT": str(TRAIN_TXT), "BC_VAL_TXT": str(TEST_TXT),
    "BC_TEST_JSON": str(PREP / "annotations" / "test_coco.json"), "BC_DATA_ROOT": PREP.as_posix() + "/",
    "BC_SAVE_DIR": str(RUNS), "BC_EPOCHS": str(EPOCHS), "BC_WORKERS": str(WORKERS),
})
print(f"patched; {n_tr} train and {n_te} test samples")

## 6. Quick check before the long run
Builds the model, loads the YOLOX-s weights, and trains a few real batches. It picks a batch size that fits, measures speed, and estimates the total time. Skipped on later sessions, which reuse the saved run settings.

In [ ]:
SMOKE = r'''
import json, os, sys, time
import numpy as np, torch
from torch.utils.data import DataLoader
from nets.BeltCrackNet import slowfastnet
from nets.yolo_training import YOLOLoss, weights_init
from utils.dataloader_for_CRACK import seqDataset, dataset_collate

batch, fp16 = int(os.environ["BC_BATCH"]), os.environ["BC_FP16"] == "1"
torch.manual_seed(0)
model = slowfastnet(num_classes=1, num_frames=5)
weights_init(model)
sd = model.state_dict()
pre = torch.load("model_data/yolox_s.pth", map_location="cpu")
ok = {k: v for k, v in pre.items() if k in sd and np.shape(sd[k]) == np.shape(v)}
sd.update(ok)
model.load_state_dict(sd)
bb = [k for k in sd if k.startswith("backbone.")]
ds = seqDataset(os.environ["BC_TRAIN_TXT"], 512, 5, "train")
x0, b0 = ds[0]
dl = DataLoader(ds, batch_size=batch, shuffle=True, num_workers=int(os.environ["BC_WORKERS"]),
                collate_fn=dataset_collate, drop_last=True)
model = model.cuda().train()
loss_fn = YOLOLoss(1, fp16, strides=[8])
opt = torch.optim.SGD(model.parameters(), lr=1e-4, momentum=0.9)
scaler = torch.cuda.amp.GradScaler() if fp16 else None
torch.cuda.reset_peak_memory_stats()
times, it = [], iter(dl)
try:
    for _ in range(8):
        t0 = time.time()
        images, targets = next(it)
        images, targets = images.cuda(), [t.cuda() for t in targets]
        opt.zero_grad()
        if fp16:
            with torch.cuda.amp.autocast():
                loss = loss_fn(model(images), targets)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        else:
            loss = loss_fn(model(images), targets)
            loss.backward(); opt.step()
        torch.cuda.synchronize()
        times.append(time.time() - t0)
        assert torch.isfinite(loss), "loss is not finite"
except torch.cuda.OutOfMemoryError:
    print("SMOKE_OOM"); sys.exit(3)
print("SMOKE_RESULT " + json.dumps({
    "batch": batch, "sample_shape": list(x0.shape), "boxes_first_sample": int(len(b0)),
    "pretrained_keys_loaded": len(ok), "backbone_keys_loaded": sum(k in ok for k in bb), "backbone_keys": len(bb),
    "sec_per_step": float(np.median(times[3:])), "peak_gb": torch.cuda.max_memory_allocated() / 2**30,
    "last_loss": float(loss)}))
'''

CONFIG = STORE / "run_config.json"
done, _ = find_resume(RUNS)
if CONFIG.exists() and done > 0:
    cfg = json.loads(CONFIG.read_text())
    if (cfg["fp16"], cfg["epochs"]) != (FP16, EPOCHS):
        print(f"NOTE: continuing the existing run with its own settings {cfg}, not the ones in section 1.")
    print("existing run found at epoch", done, "- skipping the check")
else:
    (REPO / "_smoke_test.py").write_text(SMOKE)
    batch, res = BATCH, None
    while batch >= 2 and res is None:
        env = dict(os.environ, BC_BATCH=str(batch), BC_FP16="1" if FP16 else "0")
        r = subprocess.run([sys.executable, "_smoke_test.py"], cwd=REPO, env=env, capture_output=True, text=True)
        out = r.stdout + r.stderr
        m = re.search(r"SMOKE_RESULT (\{.*\})", out)
        if m:
            res = json.loads(m.group(1))
        elif "SMOKE_OOM" in out or "out of memory" in out.lower():
            print(f"batch {batch} does not fit in GPU memory, trying {batch // 2}")
            batch //= 2
        else:
            raise RuntimeError("check failed:\n" + out[-4000:])
    if res is None:
        raise RuntimeError("even batch 2 does not fit in GPU memory")
    print(json.dumps(res, indent=1))
    assert res["backbone_keys_loaded"] == res["backbone_keys"], "pretrained backbone did not load fully"
    if batch != BATCH:
        print(f"NOTE: using batch {batch} instead of {BATCH}. The authors' code scales the learning rate by batch size.")
    steps = INFO["train"]["images"] // batch
    val_steps = INFO["test"]["images"] // batch
    per_epoch = steps * res["sec_per_step"] + val_steps * res["sec_per_step"] * 0.35
    evals = EPOCHS // 5 * INFO["test"]["images"] * res["sec_per_step"] / batch * 0.5
    total_h = (EPOCHS * per_epoch + evals) / 3600
    print(f"rough estimate: {per_epoch / 60:.0f} min per epoch, about {total_h:.0f} h for {EPOCHS} epochs on this GPU")
    cfg = {"batch": batch, "fp16": FP16, "epochs": EPOCHS, "repo_commit": REPO_COMMIT, "smoke": res}
    CONFIG.write_text(json.dumps(cfg, indent=1))

os.environ.update({"BC_BATCH": str(cfg["batch"]), "BC_FP16": "1" if cfg["fp16"] else "0", "BC_EPOCHS": str(cfg["epochs"])})
EPOCHS_RUN = cfg["epochs"]

## 7. Train (resumes automatically)
Checkpoints go to `runs/` in your Drive folder: every epoch (`last_epoch_weights.pth`) and every 5 epochs (`ep00NN-....pth`). Every 5 epochs the authors' callback also computes mAP on the test split.

In [ ]:
done, weights = find_resume(RUNS)
if done >= EPOCHS_RUN:
    print(f"training finished ({done} epochs)")
else:
    os.environ["BC_INIT_EPOCH"] = str(done)
    os.environ["BC_MODEL_PATH"] = str(weights) if weights else "model_data/yolox_s.pth"
    print(f"{'resuming after epoch ' + str(done) if done else 'starting'} from {os.environ['BC_MODEL_PATH']}")
    !cd "{REPO}" && python train_BeltCrackDet.py

## 8. Training curve (mAP50 on the test split, from the authors' callback)

In [ ]:
import matplotlib.pyplot as plt

maps = read_epoch_maps(RUNS)
if maps:
    ep = list(maps)
    plt.figure(figsize=(7, 4))
    plt.plot(ep, [maps[e][0] for e in ep], "o-", label="mAP50")
    plt.plot(ep, [maps[e][3] for e in ep], "s--", label="F1")
    plt.axhline(0.3079, color="gray", ls=":", label="paper: 30.79 mAP50")
    plt.xlabel("epoch")
    plt.legend()
    plt.grid(alpha=.3)
    plt.show()
    for e in ep:
        print(f"epoch {e:3d}: mAP50 {maps[e][0]:.4f}  P {maps[e][1]:.4f}  R {maps[e][2]:.4f}  F1 {maps[e][3]:.4f}")
else:
    print("no evaluation yet (the callback runs every 5 epochs)")

## 9. Final test evaluation with the authors' `vid_map_coco.py`

In [ ]:
def pick_weights(name):
    if name == "last":
        done, w = find_resume(RUNS)
        if w is None:
            raise RuntimeError("no checkpoint yet")
        if done < EPOCHS_RUN:
            print(f"NOTE: training is at epoch {done} of {EPOCHS_RUN}; evaluating that checkpoint")
        return w
    hits = sorted(RUNS.glob(f"loss_*/{name}"))
    if not hits:
        raise RuntimeError(f"{name} not found under {RUNS}")
    return hits[-1]


def evaluate(weights, out_dir):
    env = dict(os.environ, BC_EVAL_WEIGHTS=str(weights), BC_EVAL_OUT=str(out_dir),
               BC_EVAL_CONF=str(EVAL_CONF), BC_EVAL_NMS=str(EVAL_NMS))
    r = subprocess.run([sys.executable, "vid_map_coco.py"], cwd=REPO, env=env, capture_output=True, text=True)
    out = r.stdout + r.stderr
    if r.returncode:
        raise RuntimeError(out[-4000:])

    def ap(iou):
        m = re.search(rf"Average Precision\s+\(AP\) @\[ IoU={re.escape(iou)}\s*\| area=\s*all"
                      r" \| maxDets=100 \] = (-?[\d.]+)", out)
        return float(m.group(1)) if m else None

    m = re.search(r"Precision: ([\d.nan]+), Recall: ([\d.nan]+), F1: ([\d.nan]+)", out)
    return {"weights": str(weights), "mAP50": ap("0.50"), "mAP50_95": ap("0.50:0.95"),
            "precision": float(m.group(1)) if m else None, "recall": float(m.group(2)) if m else None,
            "F1": float(m.group(3)) if m else None, "conf": EVAL_CONF, "nms": EVAL_NMS}


W = pick_weights(EVAL_CHECKPOINT)
EVAL_OUT = BASE / "eval_out"
result = evaluate(W, EVAL_OUT)
result["paper_mAP50_9kd"] = 0.3079
(STORE / "test_metrics.json").write_text(json.dumps(result, indent=1))
print(json.dumps(result, indent=1))

## 10. Optional: evaluate every saved checkpoint
This reproduces the authors' advice to pick the best file from `results/`. On 9kd that choice is made on the **test** split, so the best number here is optimistic; report the section 9 number as the main result.

In [ ]:
RUN_ALL_CHECKPOINTS = False

if RUN_ALL_CHECKPOINTS:
    rows = []
    for w in sorted(RUNS.glob("loss_*/ep*.pth"), key=lambda p: p.name):
        r = evaluate(w, BASE / "eval_all" / w.stem)
        rows.append(r)
        print(f"{w.name}: mAP50 {r['mAP50']}  P {r['precision']}  R {r['recall']}")
    (STORE / "all_checkpoints.json").write_text(json.dumps(rows, indent=1))

## 11. Look at some test predictions
Green: ground truth. Red: predictions with score ≥ 0.3.

In [ ]:
import random

from PIL import Image, ImageDraw

gt = json.loads((PREP / "annotations" / "test_coco.json").read_text())
dets = json.loads((EVAL_OUT / "eval_results.json").read_text())
by_img = {}
for d in dets:
    by_img.setdefault(d["image_id"], []).append(d)
anns = {}
for a in gt["annotations"]:
    anns.setdefault(a["image_id"], []).append(a)
pick = random.Random(0).sample(gt["images"], min(6, len(gt["images"])))
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, im in zip(axes.ravel(), pick):
    img = Image.open(PREP / im["file_name"]).convert("RGB")
    dr = ImageDraw.Draw(img)
    for a in anns.get(im["id"], []):
        x, y, w, h = a["bbox"]
        dr.rectangle([x, y, x + w, y + h], outline=(0, 255, 0), width=3)
    for d in by_img.get(im["id"], []):
        if d["score"] >= 0.3:
            x, y, w, h = d["bbox"]
            dr.rectangle([x, y, x + w, y + h], outline=(255, 0, 0), width=2)
    ax.imshow(img)
    ax.set_title(im["file_name"], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()